## Phase 4 — Mesh Stitching

**Input:** Segment mesh files (`.obj`/`.glb`) + `segments_meta.json` from Phase 1.  
**Output:** Unified mesh (`unified_mesh.glb`) in the output directory.  

**How to run:** Execute all cells top to bottom. Upload mesh files and `segments_meta.json` when prompted (Colab), or set paths in cell 2.

In [ ]:
import subprocess
import sys

print('Installing dependencies...')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'numpy'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'scipy'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'trimesh'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyglet', 'networkx'], check=True)

print('Dependencies installed!')

In [ ]:
import os

MESHES_DIR = '/content/meshes'  # @param {type:"string"}

SEGMENTS_META_PATH = '/content/segments_meta.json'  # @param {type:"string"}

UPLOAD_FILES = True  # @param {type:"boolean"}

OUTPUT_DIR = '/content/stage8_9_output'  # @param {type:"string"}
OUTPUT_FILENAME = 'unified_mesh.glb'  # @param {type:"string"}

WELD_THRESHOLD = 0.01  # @param {type:"slider", min:0.001, max:0.1, step:0.001}

BOUNDARY_MARGIN = 0.05  # @param {type:"slider", min:0.01, max:0.2, step:0.01}

SMOOTH_BOUNDARIES = True  # @param {type:"boolean"}

SMOOTH_ITERATIONS = 2  # @param {type:"slider", min:1, max:5, step:1}

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MESHES_DIR, exist_ok=True)
print('Configuration loaded')

In [ ]:
import json

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if UPLOAD_FILES and IN_COLAB:
    print('Please upload your mesh files (.obj or .glb):')
    uploaded_meshes = files.upload()
    for fname in uploaded_meshes.keys():
        with open(os.path.join(MESHES_DIR, fname), 'wb') as f:
            f.write(uploaded_meshes[fname])
    print(f'Uploaded {len(uploaded_meshes)} mesh files')

    print('\nPlease upload segments_meta.json:')
    uploaded_meta = files.upload()
    meta_fname = list(uploaded_meta.keys())[0]
    with open(SEGMENTS_META_PATH, 'wb') as f:
        f.write(uploaded_meta[meta_fname])
    print('Metadata uploaded')
else:
    print(f'Using mesh files from: {MESHES_DIR}')
    print(f'Using metadata from: {SEGMENTS_META_PATH}')

In [ ]:
import json

with open(SEGMENTS_META_PATH, 'r') as f:
    segments_meta = json.load(f)

if isinstance(segments_meta, list):
    segments_list = segments_meta
elif isinstance(segments_meta, dict) and 'segments' in segments_meta:
    segments_list = segments_meta['segments']
else:
    segments_list = []

print(f'Loaded metadata for {len(segments_list)} segments')
print()

for seg in segments_list:
    seg_id = seg['segment_id']
    bbox = seg['dxf_bbox']
    neighbors = seg.get('neighbors', [])
    print(f'  Segment {seg_id}:')
    print(f'    DXF BBox: ({bbox["min_x"]:.2f}, {bbox["min_y"]:.2f}) to ({bbox["max_x"]:.2f}, {bbox["max_y"]:.2f})')
    print(f'    Neighbors: {neighbors}')
    print()

In [ ]:
import trimesh
import numpy as np
import re
import glob

mesh_files = glob.glob(os.path.join(MESHES_DIR, '*.obj')) + \
             glob.glob(os.path.join(MESHES_DIR, '*.glb')) + \
             glob.glob(os.path.join(MESHES_DIR, '*.gltf'))

print(f'Found {len(mesh_files)} mesh files')

seg_by_num = {}
for seg in segments_list:
    m = re.search(r'(\d+)', str(seg['segment_id']))
    if m:
        seg_by_num[int(m.group(1))] = seg

seen_base_nums = {}
deduped_files = []
for mesh_path in sorted(mesh_files, key=lambda p: len(os.path.basename(p))):
    fname = os.path.basename(mesh_path)
    base = re.sub(r'\s*\(\d+\)', '', fname)
    m = re.search(r'(\d+)', base)
    if m:
        num = int(m.group(1))
        if num not in seen_base_nums:
            seen_base_nums[num] = fname
            deduped_files.append(mesh_path)
        else:
            print(f'  Skipping duplicate: {fname} (same as {seen_base_nums[num]})')
    else:
        deduped_files.append(mesh_path)

print(f'After dedup: {len(deduped_files)} unique mesh files')

meshes = {}
for mesh_path in deduped_files:
    fname = os.path.basename(mesh_path)

    seg_id = None
    for seg in segments_list:
        sid = str(seg['segment_id'])
        if sid in fname or seg.get('mesh_file', '') == fname:
            seg_id = sid
            break

    if seg_id is None:
        base = re.sub(r'\s*\(\d+\)', '', fname)
        m = re.search(r'(\d+)', base)
        if m:
            num = int(m.group(1))
            if num in seg_by_num:
                seg_id = seg_by_num[num]['segment_id']
            else:
                seg_id = f'unmapped_{num}'
        else:
            seg_id = f'unmapped_{len(meshes)}'

    if not segments_list and seg_id.startswith('unmapped_'):
        seg_id = f'seg_{len(meshes):04d}'

    print(f'  Loading {fname} -> {seg_id}')

    loaded = trimesh.load(mesh_path)
    if isinstance(loaded, trimesh.Scene):
        parts = [g for g in loaded.geometry.values() if isinstance(g, trimesh.Trimesh)]
        if parts:
            mesh = trimesh.util.concatenate(parts)
        else:
            print(f'    Warning: No geometry in {fname}, skipping')
            continue
    else:
        mesh = loaded

    meshes[seg_id] = {
        'mesh': mesh,
        'file': fname,
        'vertices': len(mesh.vertices),
        'faces': len(mesh.faces)
    }

print(f'\nLoaded {len(meshes)} meshes')
for seg_id, data in meshes.items():
    print(f'  {seg_id}: {data["vertices"]} vertices, {data["faces"]} faces')

In [ ]:
import numpy as np

all_xs, all_ys = [], []
for seg in segments_list:
    bbox = seg['dxf_bbox']
    all_xs.extend([bbox['min_x'], bbox['max_x']])
    all_ys.extend([bbox['min_y'], bbox['max_y']])

if all_xs:
    GLOBAL_OFFSET = np.array([(min(all_xs) + max(all_xs)) / 2,
                               (min(all_ys) + max(all_ys)) / 2, 0.0])
    global_extent = max(max(all_xs) - min(all_xs), max(all_ys) - min(all_ys))
    AUTO_BOUNDARY_MARGIN = global_extent * 0.02    # 2% of scene extent
    AUTO_WELD_THRESHOLD  = global_extent * 0.001   # 0.1% of scene extent
    print(f'Global offset: ({GLOBAL_OFFSET[0]:.1f}, {GLOBAL_OFFSET[1]:.1f})')
    print(f'Scene extent: {global_extent:.1f}')
    print(f'Auto boundary margin: {AUTO_BOUNDARY_MARGIN:.2f}')
    print(f'Auto weld threshold: {AUTO_WELD_THRESHOLD:.4f}')
else:
    GLOBAL_OFFSET = np.array([0.0, 0.0, 0.0])
    AUTO_BOUNDARY_MARGIN = BOUNDARY_MARGIN
    AUTO_WELD_THRESHOLD = WELD_THRESHOLD
    print('No segments metadata — using config thresholds')

def transform_to_global(mesh, dxf_bbox, offset):
    """
    Transform mesh from local normalized space to global DXF coordinates,
    centered near origin by subtracting offset.
    Uses float64 to avoid precision collapse at large DXF coordinates.
    """
    vertices = mesh.vertices.copy().astype(np.float64)

    mesh_min = vertices.min(axis=0)
    mesh_max = vertices.max(axis=0)
    mesh_size = mesh_max - mesh_min

    dxf_min = np.array([dxf_bbox['min_x'], dxf_bbox['min_y'], 0.0], dtype=np.float64)
    dxf_max = np.array([dxf_bbox['max_x'], dxf_bbox['max_y'], 0.0], dtype=np.float64)
    dxf_size = dxf_max - dxf_min

    xy_scale = max(dxf_size[0], dxf_size[1]) / max(mesh_size[0], mesh_size[1]) if max(mesh_size[0], mesh_size[1]) > 0 else 1.0
    dxf_size[2] = mesh_size[2] * xy_scale

    if np.any(mesh_size > 0):
        vertices = (vertices - mesh_min) / np.where(mesh_size > 0, mesh_size, 1.0)

    vertices = vertices * dxf_size + dxf_min - offset

    transformed = trimesh.Trimesh(
        vertices=vertices,
        faces=mesh.faces.copy(),
        process=False
    )

    return transformed

transformed_meshes = {}

for seg in segments_list:
    seg_id = seg['segment_id']
    if seg_id not in meshes:
        print(f'  Warning: No mesh found for segment {seg_id}')
        continue

    dxf_bbox = seg['dxf_bbox']
    original_mesh = meshes[seg_id]['mesh']

    transformed = transform_to_global(original_mesh, dxf_bbox, GLOBAL_OFFSET)
    transformed_meshes[seg_id] = {
        'mesh': transformed,
        'dxf_bbox': dxf_bbox,
        'neighbors': seg.get('neighbors', [])
    }

    print(f'  Segment {seg_id}: transformed to DXF coords (centered)')
    print(f'    Bounds: ({transformed.bounds[0][0]:.2f}, {transformed.bounds[0][1]:.2f}) to ({transformed.bounds[1][0]:.2f}, {transformed.bounds[1][1]:.2f})')

for seg_id, data in meshes.items():
    if seg_id not in transformed_meshes:
        print(f'  Segment {seg_id}: no metadata, keeping original coordinates')
        transformed_meshes[seg_id] = {
            'mesh': data['mesh'],
            'dxf_bbox': None,
            'neighbors': []
        }

print(f'\nTotal meshes ready: {len(transformed_meshes)}')

In [ ]:
@title Cell 6b: Resolve Mesh Overlaps
SAM segments often have overlapping bounding boxes.
When we stretch each mesh to fill its DXF bbox, the meshes physically intersect.
Fix: clip each mesh at the midpoint of the overlap zone so they meet without interfering.
def clip_mesh_to_halfspace(mesh, plane_origin, plane_normal):
"""Keep the half of the mesh on the positive side of the plane."""
plane_normal = np.array(plane_normal, dtype=np.float64)
plane_origin = np.array(plane_origin, dtype=np.float64)
try:
clipped = mesh.slice_mesh_plane(plane_origin, plane_normal, cap=False)
if clipped is not None and len(clipped.vertices) > 0:
return clipped
except Exception:
pass
Fallback: remove faces entirely on the negative side
dots = np.dot(mesh.vertices.astype(np.float64) - plane_origin, plane_normal)
face_dots = dots[mesh.faces]
keep_mask = np.any(face_dots >= 0, axis=1)
if keep_mask.all():
return mesh
new_mesh = mesh.copy()
new_mesh.update_faces(keep_mask)
new_mesh.remove_unreferenced_vertices()
return new_mesh
print('Resolving mesh overlaps...')
overlap_pairs = set()
for seg_id, data in list(transformed_meshes.items()):
bbox1 = data.get('dxf_bbox')
if bbox1 is None:
continue
for neighbor_id in data.get('neighbors', []):
pair = tuple(sorted([str(seg_id), str(neighbor_id)]))
if pair in overlap_pairs:
continue
overlap_pairs.add(pair)
if neighbor_id not in transformed_meshes:
continue
neighbor_data = transformed_meshes[neighbor_id]
bbox2 = neighbor_data.get('dxf_bbox')
if bbox2 is None:
continue
Check overlap in DXF coords
x_overlap = min(bbox1['max_x'], bbox2['max_x']) - max(bbox1['min_x'], bbox2['min_x'])
y_overlap = min(bbox1['max_y'], bbox2['max_y']) - max(bbox1['min_y'], bbox2['min_y'])
if x_overlap <= 0 or y_overlap <= 0:
print(f' {seg_id} <-> {neighbor_id}: no overlap, OK')
continue
print(f' {seg_id} <-> {neighbor_id}: overlap {x_overlap:.0f} x {y_overlap:.0f} units')
Determine dominant separation direction via bbox centers
cx1 = (bbox1['min_x'] + bbox1['max_x']) / 2
cx2 = (bbox2['min_x'] + bbox2['max_x']) / 2
cy1 = (bbox1['min_y'] + bbox1['max_y']) / 2
cy2 = (bbox2['min_y'] + bbox2['max_y']) / 2
dx = abs(cx1 - cx2)
dy = abs(cy1 - cy2)
if dx >= dy:
Split along X at midpoint of overlap zone
overlap_min = max(bbox1['min_x'], bbox2['min_x'])
overlap_max = min(bbox1['max_x'], bbox2['max_x'])
split = (overlap_min + overlap_max) / 2 - GLOBAL_OFFSET[0]
plane_origin = [split, 0, 0]
axis_label = 'X'
if cx1 > cx2:
normal_seg = [1, 0, 0]
normal_neighbor = [-1, 0, 0]
else:
normal_seg = [-1, 0, 0]
normal_neighbor = [1, 0, 0]
else:
Split along Y at midpoint of overlap zone
overlap_min = max(bbox1['min_y'], bbox2['min_y'])
overlap_max = min(bbox1['max_y'], bbox2['max_y'])
split = (overlap_min + overlap_max) / 2 - GLOBAL_OFFSET[1]
plane_origin = [0, split, 0]
axis_label = 'Y'
if cy1 > cy2:
normal_seg = [0, 1, 0]
normal_neighbor = [0, -1, 0]
else:
normal_seg = [0, -1, 0]
normal_neighbor = [0, 1, 0]
print(f' Cutting along {axis_label} at {split:.1f} (centered coords)')
Clip both meshes
mesh1_before = len(data['mesh'].faces)
mesh2_before = len(neighbor_data['mesh'].faces)
data['mesh'] = clip_mesh_to_halfspace(data['mesh'], plane_origin, normal_seg)
neighbor_data['mesh'] = clip_mesh_to_halfspace(neighbor_data['mesh'], plane_origin, normal_neighbor)
print(f' {seg_id}: {mesh1_before} -> {len(data["mesh"].faces)} faces')
print(f' {neighbor_id}: {mesh2_before} -> {len(neighbor_data["mesh"].faces)} faces')
Update dxf_bbox to match actual clipped bounds (for boundary detection)
for sid, sdata in [(seg_id, data), (neighbor_id, neighbor_data)]:
b = sdata['mesh'].bounds
sdata['dxf_bbox'] = {
'min_x': float(b[0][0]) + GLOBAL_OFFSET[0],
'min_y': float(b[0][1]) + GLOBAL_OFFSET[1],
'max_x': float(b[1][0]) + GLOBAL_OFFSET[0],
'max_y': float(b[1][1]) + GLOBAL_OFFSET[1]
}
print('Overlap resolution complete')
for seg_id, data in transformed_meshes.items():
m = data['mesh']
print(f' {seg_id}: {len(m.vertices)} verts, {len(m.faces)} faces, '
f'bounds ({m.bounds[0][0]:.1f}, {m.bounds[0][1]:.1f}) to ({m.bounds[1][0]:.1f}, {m.bounds[1][1]:.1f})')

In [ ]:
def get_boundary_vertices(mesh, dxf_bbox, margin, offset):
    """
    Find vertices near the edges of the DXF bounding box (after offset centering).
    These are candidates for stitching with neighbors.
    """
    vertices = mesh.vertices

    min_x = dxf_bbox['min_x'] - offset[0]
    min_y = dxf_bbox['min_y'] - offset[1]
    max_x = dxf_bbox['max_x'] - offset[0]
    max_y = dxf_bbox['max_y'] - offset[1]

    boundary_indices = {
        'left':   np.where(np.abs(vertices[:, 0] - min_x) < margin)[0],
        'right':  np.where(np.abs(vertices[:, 0] - max_x) < margin)[0],
        'bottom': np.where(np.abs(vertices[:, 1] - min_y) < margin)[0],
        'top':    np.where(np.abs(vertices[:, 1] - max_y) < margin)[0]
    }

    all_boundary = np.unique(np.concatenate([
        boundary_indices['left'],
        boundary_indices['right'],
        boundary_indices['bottom'],
        boundary_indices['top']
    ])) if any(len(v) > 0 for v in boundary_indices.values()) else np.array([], dtype=int)

    return boundary_indices, all_boundary

for seg_id, data in transformed_meshes.items():
    mesh = data['mesh']
    dxf_bbox = data.get('dxf_bbox')

    if dxf_bbox is None:
        data['boundary_indices'] = {'left': np.array([]), 'right': np.array([]), 'bottom': np.array([]), 'top': np.array([])}
        data['all_boundary'] = np.array([], dtype=int)
        print(f'  {seg_id}: no bbox, skipping boundary detection')
        continue

    boundary_indices, all_boundary = get_boundary_vertices(mesh, dxf_bbox, AUTO_BOUNDARY_MARGIN, GLOBAL_OFFSET)
    data['boundary_indices'] = boundary_indices
    data['all_boundary'] = all_boundary

    print(f'  {seg_id}: {len(all_boundary)} boundary vertices')
    print(f'    Left: {len(boundary_indices["left"])}, Right: {len(boundary_indices["right"])}')
    print(f'    Bottom: {len(boundary_indices["bottom"])}, Top: {len(boundary_indices["top"])}')

In [ ]:
from scipy.spatial import cKDTree

def align_boundaries(mesh1, mesh2, boundary1, boundary2, threshold):
    """
    Align boundary vertices of mesh2 to match mesh1 where they overlap.
    Returns modified vertices for mesh2.
    """
    verts1 = mesh1.vertices[boundary1]
    verts2 = mesh2.vertices.copy()

    if len(verts1) == 0 or len(boundary2) == 0:
        return verts2, 0

    tree = cKDTree(verts1)

    aligned_count = 0
    for idx in boundary2:
        dist, nearest_idx = tree.query(verts2[idx])
        if dist < threshold:
            verts2[idx] = verts1[nearest_idx]
            aligned_count += 1

    return verts2, aligned_count

def determine_shared_edge(bbox1, bbox2, margin):
    """
    Determine which edges are shared between two bboxes.
    Handles both touching and overlapping bboxes by comparing centers.
    All bbox coordinates are already offset-centered.
    """
    edges1 = []
    edges2 = []

    x_overlap = min(bbox1['max_x'], bbox2['max_x']) - max(bbox1['min_x'], bbox2['min_x'])
    y_overlap = min(bbox1['max_y'], bbox2['max_y']) - max(bbox1['min_y'], bbox2['min_y'])

    if x_overlap > -margin and y_overlap > -margin:
        cx1 = (bbox1['min_x'] + bbox1['max_x']) / 2
        cx2 = (bbox2['min_x'] + bbox2['max_x']) / 2
        cy1 = (bbox1['min_y'] + bbox1['max_y']) / 2
        cy2 = (bbox2['min_y'] + bbox2['max_y']) / 2

        dx = abs(cx1 - cx2)
        dy = abs(cy1 - cy2)

        if dx >= dy:
            if cx1 < cx2:
                edges1.append('right')
                edges2.append('left')
            else:
                edges1.append('left')
                edges2.append('right')
        else:
            if cy1 < cy2:
                edges1.append('top')
                edges2.append('bottom')
            else:
                edges1.append('bottom')
                edges2.append('top')

    return edges1, edges2

print('Aligning boundary vertices between neighbors...')
weld_thresh = AUTO_WELD_THRESHOLD

total_aligned = 0
processed_pairs = set()

for seg_id, data in transformed_meshes.items():
    for neighbor_id in data['neighbors']:
        pair = tuple(sorted([str(seg_id), str(neighbor_id)]))
        if pair in processed_pairs:
            continue
        processed_pairs.add(pair)

        if neighbor_id not in transformed_meshes:
            print(f'  Warning: Neighbor {neighbor_id} not found for segment {seg_id}')
            continue

        neighbor_data = transformed_meshes[neighbor_id]

        if data.get('dxf_bbox') is None or neighbor_data.get('dxf_bbox') is None:
            continue

        edges1, edges2 = determine_shared_edge(data['dxf_bbox'], neighbor_data['dxf_bbox'], AUTO_BOUNDARY_MARGIN)

        if not edges1:
            print(f'  {seg_id} <-> {neighbor_id}: no shared edge found')
            continue

        print(f'  {seg_id} ({edges1}) <-> {neighbor_id} ({edges2})')

        boundary1_list = [data['boundary_indices'].get(e, np.array([])) for e in edges1]
        boundary2_list = [neighbor_data['boundary_indices'].get(e, np.array([])) for e in edges2]
        boundary1 = np.concatenate(boundary1_list).astype(int) if boundary1_list else np.array([], dtype=int)
        boundary2 = np.concatenate(boundary2_list).astype(int) if boundary2_list else np.array([], dtype=int)

        new_verts, aligned = align_boundaries(
            data['mesh'],
            neighbor_data['mesh'],
            boundary1,
            boundary2,
            weld_thresh
        )

        neighbor_data['mesh'] = trimesh.Trimesh(
            vertices=new_verts,
            faces=neighbor_data['mesh'].faces,
            process=False
        )

        if aligned > 0:
            print(f'    Aligned {aligned} vertices')
        else:
            print(f'    0 vertices within weld threshold ({weld_thresh:.4f})')
        total_aligned += aligned

print(f'\nTotal aligned vertices: {total_aligned}')

In [ ]:
print('Combining all meshes...')

all_meshes = [data['mesh'] for data in transformed_meshes.values()]

if not all_meshes:
    print('WARNING: No meshes to combine!')
    combined_mesh = trimesh.Trimesh()
else:
    combined_mesh = trimesh.util.concatenate(all_meshes)

print(f'Combined mesh:')
print(f'  Vertices: {len(combined_mesh.vertices)}')
print(f'  Faces: {len(combined_mesh.faces)}')

In [ ]:
if len(combined_mesh.vertices) == 0:
    print('Skipping weld — empty mesh')
    pre_weld_verts = 0
    post_weld_verts = 0
else:
    print(f'Welding vertices within {AUTO_WELD_THRESHOLD:.4f} units...')

    pre_weld_verts = len(combined_mesh.vertices)
    combined_mesh.merge_vertices(merge_tex=True, merge_norm=True)
    post_weld_verts = len(combined_mesh.vertices)

    print(f'  Before: {pre_weld_verts} vertices')
    print(f'  After:  {post_weld_verts} vertices')
    print(f'  Welded: {pre_weld_verts - post_weld_verts} duplicate vertices')

In [ ]:
if not SMOOTH_BOUNDARIES or len(combined_mesh.vertices) == 0:
    if not SMOOTH_BOUNDARIES:
        print('Boundary smoothing disabled')
    else:
        print('Skipping smoothing — empty mesh')
else:
    print(f'Smoothing boundary regions ({SMOOTH_ITERATIONS} iterations)...')

    vertices = combined_mesh.vertices.copy()
    faces = combined_mesh.faces
    n_verts = len(vertices)

    adjacency = [set() for _ in range(n_verts)]
    for face in faces:
        for i in range(3):
            adjacency[face[i]].add(face[(i+1)%3])
            adjacency[face[i]].add(face[(i+2)%3])

    edge_count = {}
    for face in faces:
        for i in range(3):
            edge = tuple(sorted([face[i], face[(i+1)%3]]))
            edge_count[edge] = edge_count.get(edge, 0) + 1

    boundary_verts = set()
    for edge, count in edge_count.items():
        if count == 1:
            boundary_verts.add(edge[0])
            boundary_verts.add(edge[1])

    margin = AUTO_BOUNDARY_MARGIN
    for seg_id, data in transformed_meshes.items():
        bbox = data.get('dxf_bbox')
        if bbox is None:
            continue
        bmin_x = bbox['min_x'] - GLOBAL_OFFSET[0]
        bmax_x = bbox['max_x'] - GLOBAL_OFFSET[0]
        bmin_y = bbox['min_y'] - GLOBAL_OFFSET[1]
        bmax_y = bbox['max_y'] - GLOBAL_OFFSET[1]

        near_edge = (
            (np.abs(vertices[:, 0] - bmin_x) < margin) |
            (np.abs(vertices[:, 0] - bmax_x) < margin) |
            (np.abs(vertices[:, 1] - bmin_y) < margin) |
            (np.abs(vertices[:, 1] - bmax_y) < margin)
        )
        boundary_verts.update(np.where(near_edge)[0])

    print(f'  Found {len(boundary_verts)} vertices to smooth')

    for iteration in range(SMOOTH_ITERATIONS):
        new_vertices = vertices.copy()
        for v_idx in boundary_verts:
            neighbors = list(adjacency[v_idx])
            if len(neighbors) > 0:
                avg = vertices[neighbors].mean(axis=0)
                new_vertices[v_idx] = 0.5 * vertices[v_idx] + 0.5 * avg
        vertices = new_vertices

    combined_mesh = trimesh.Trimesh(
        vertices=vertices,
        faces=combined_mesh.faces,
        process=False
    )

    print('  Smoothing complete')

In [ ]:
if len(combined_mesh.faces) == 0:
    print('Skipping cleanup — empty mesh')
    pre_clean = 0
    post_clean = 0
else:
    print('Cleaning up mesh...')

    pre_clean = len(combined_mesh.faces)

    face_mask = combined_mesh.nondegenerate_faces()
    if not face_mask.all():
        combined_mesh.update_faces(face_mask)
        print(f'  Removed {(~face_mask).sum()} degenerate faces')

    combined_mesh.update_faces(combined_mesh.unique_faces())
    combined_mesh.remove_unreferenced_vertices()

    post_clean = len(combined_mesh.faces)
    print(f'  Total faces removed: {pre_clean - post_clean}')

    combined_mesh.fix_normals()

    print(f'\nFinal mesh:')
    print(f'  Vertices: {len(combined_mesh.vertices)}')
    print(f'  Faces: {len(combined_mesh.faces)}')
    print(f'  Watertight: {combined_mesh.is_watertight}')
    if combined_mesh.is_watertight:
        print(f'  Volume: {combined_mesh.volume:.2f}')

In [ ]:
output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)

combined_mesh.export(output_path, file_type='glb')

print(f'Mesh exported: {output_path}')
print(f'File size: {os.path.getsize(output_path) / 1e6:.2f} MB')

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

def render_mesh_preview(mesh, title='Unified Mesh Preview'):
    """Render a preview of the unified mesh."""
    fig = plt.figure(figsize=(14, 6))

    vertices = np.array(mesh.vertices)

    max_points = 10000
    if len(vertices) > max_points:
        idx = np.random.choice(len(vertices), max_points, replace=False)
        vertices_sub = vertices[idx]
    else:
        vertices_sub = vertices

    views = [(90, 0, 'Top View'), (0, 0, 'Front View'), (30, 45, 'Perspective')]

    for i, (elev, azim, view_title) in enumerate(views):
        ax = fig.add_subplot(1, 3, i+1, projection='3d')
        ax.scatter(vertices_sub[:, 0], vertices_sub[:, 1], vertices_sub[:, 2],
                   c=vertices_sub[:, 2], cmap='viridis', s=0.5, alpha=0.6)
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        ax.view_init(elev=elev, azim=azim)
        ax.set_title(view_title)

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    return fig

fig = render_mesh_preview(combined_mesh, 'Unified Mesh - Stages 8 & 9')

preview_path = os.path.join(OUTPUT_DIR, 'unified_mesh_preview.png')
fig.savefig(preview_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Preview saved: {preview_path}')

In [ ]:
if IN_COLAB:
    print('Downloading unified mesh...')
    files.download(output_path)

    print('Downloading preview...')
    files.download(preview_path)

    print('Downloads complete!')
else:
    print(f'Results saved to: {OUTPUT_DIR}')
    print(f'  Mesh: {output_path}')
    print(f'  Preview: {preview_path}')

In [ ]:
print('='*70)
print('  Stages 8 & 9: Mesh Stitching - Summary')
print('='*70)
print()
print('  Input:')
print(f'    Meshes directory:  {MESHES_DIR}')
print(f'    Segments metadata: {SEGMENTS_META_PATH}')
print(f'    Mesh count:        {len(transformed_meshes)}')
print()
print('  Settings:')
print(f'    Weld threshold:    {WELD_THRESHOLD}')
print(f'    Boundary margin:   {BOUNDARY_MARGIN}')
print(f'    Smooth boundaries: {SMOOTH_BOUNDARIES}')
if SMOOTH_BOUNDARIES:
    print(f'    Smooth iterations: {SMOOTH_ITERATIONS}')
print()
print('  Results:')
print(f'    Total vertices:    {len(combined_mesh.vertices)}')
print(f'    Total faces:       {len(combined_mesh.faces)}')
print(f'    Welded vertices:   {pre_weld_verts - post_weld_verts}')
print(f'    Aligned vertices:  {total_aligned}')
print(f'    Watertight:        {combined_mesh.is_watertight}')
print()
print('  Output:')
print(f'    {output_path}')
print(f'    Size: {os.path.getsize(output_path) / 1e6:.2f} MB')
print()
print('='*70)